In [1]:
# This is the code to screen job posting data
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
from datetime import datetime
import gc, json, csv, re, os, glob

In [ ]:
def read_dat_data(curFile):
    colNames = ['招聘ID','公司ID','公司名称','城市名称','公司所在区域','工作薪酬','教育要求','工作经历',
                '工作描述','职位名称','工作名称','招聘数量','发布日期','行业名称','数据来源']
    resCSV = pd.read_csv(curFile, header=None, index_col=None, names=colNames,encoding='utf-8',quoting=csv.QUOTE_NONE, sep="@!", error_bad_lines=False, engine='python')
    return resCSV

In [ ]:
# The first purge of data:
# 1. We use 工作名称 as the title to feed the ChatGPT. 
# 2. We drop all the publish data is missing.
# 3. When 工作名称 is missing, we replace it with 职位名称.
# 4. We drop title of ``part-time"。
# 5. We drop same '公司ID', '工作名称', '城市名称', '工作描述' within a month, because we treat this case as duplicates (same job posting been published multiple times)
# define the file location
dataNameTmp = "E:/Data/job_posting/Raw_data/job_posting_%s.dat"

naSum = 0
dupSum = 0

curFile = dataNameTmp%21
        
print(curFile," is Grouping Computing...")
datDf = read_dat_data(curFile)
#datDf.loc[datDf['职位名称'].str.contains('%',na=False),'职位名称'] = np.NaN
 

In [ ]:
datDf = datDf.replace(r'\N',np.NaN).dropna(subset=['发布日期'])
datDf['工作名称'] = datDf[['工作名称']].replace(r'\N',np.NaN)
datDf = datDf[datDf['数据来源'].isin(['智联招聘', '前程无忧', '拉勾网', 'BOSS直聘', '58同城', '猎聘网', '看准网', '百姓网', '拉勾网', '猎聘', '赶集网', '博才网', 'BOSS'])]
datDf


In [ ]:

datDf.loc[datDf['工作名称'].isna(),'工作名称'] = datDf.loc[datDf['工作名称'].isna(),'职位名称']
datDf = datDf[datDf['工作名称'] != "兼职"]
    
curNa = datDf.shape[0]
naSum = naSum + curNa
    
datDf['date'] = datDf['发布日期'].apply(lambda x: x[0:7])
datDf = datDf.drop_duplicates(subset=['公司ID', '工作名称', '城市名称', 'date'], keep='first').reset_index(drop=True)
    
curDup = datDf.shape[0]
dupSum = dupSum + curDup
    
print(curFile,"删除空值剩余: %s"%curNa, "去重复值剩余：%s"%curDup)

In [ ]:
# Determine the data source, we limit to Top 10 job posting websites to avoid fuzzywuzzy in the data source.
dataNameTmp = "E:/Data/job_posting/Raw_data/job_posting_%s.dat"

df_list = []

for i in range(1,3): 
    curFile = dataNameTmp%i
        
    print(curFile," is Grouping Computing...")
    datDf = read_dat_data(curFile)


    # count number of occurrences of each value in column '数据来源', generate a new column to record the count
    datDf['count'] = datDf.groupby('数据来源')['数据来源'].transform('count')

    # drop duplicates based on column '数据来源', only keep the first occurrence
    datDf = datDf.drop_duplicates(subset=['数据来源'], keep='first').reset_index(drop=True)
    datDf = datDf[['数据来源', 'count']]

    df_list.append(datDf)
        
    #atDf.to_csv('F:/Data/job_posting/processed/temp/job_res_{}.csv'.format(i), sep='?', encoding = 'utf_8_sig', index=False)
final_df = pd.concat(df_list)

# group by '数据来源' and sum the count
final_df = final_df.groupby('数据来源').sum().reset_index()

# sort the dataframe based on column 'count'
final_df = final_df.sort_values(by=['count'], ascending=False)
final_df.head(20)



In [ ]:
# Append all the character data together, then generate a list of the job titles that used to feed to the ChatGPT

os.chdir("E:/Data/job_posting/processed/charac")
extension = 'csv'
all_filenames = [i for i in glob.glob('*.{}'.format(extension))]
#combine all files in the list
combined_csv = pd.concat([pd.read_csv(f, encoding = "utf_8_sig", on_bad_lines='skip', usecols = ['工作名称']) for f in all_filenames], ignore_index=True)
# This is the complete list of job posting titles 
combined_csv.to_csv('E:/Data/job_posting/processed/estimation/charac_posting.csv', index=False, header=True)
combined_csv

In [ ]:
# count the occurrences of each unique value in the 工作名称 column of a DataFrame df. The result is stored in a new variable called df_counted
df_counted = combined_csv['工作名称'].value_counts()
df_counted

In [ ]:

# Among the 99,315,582 total job postings, there is 35,537,095 job posting titles. 
df_filtered = df_counted[df_counted>4]
# Filter to values > 2, this is doable. This means we need to use ChatGPT to parse 3,771,823 job posting titles: this is equal to 60,414,820 total job postings
df_filtered.shape
# Filter to value > 100. This means we need to use ChatGPT to parse 883,695 job posting titles: this is equal to 50,340,840 total job postings



In [ ]:
# this is equal to 60,414,820 total job postings
df_filtered.sum()

In [ ]:

# convert 'df_filtered' to a dataframe including a title column and a count column
df_filtered = df_filtered.to_frame().reset_index()
df_filtered.columns = ['工作名称', 'count']

In [ ]:
# subset dataframe based on column '工作名称' equals to '兼职'
dataNameTmp = "F:/Data/job_posting/Raw_data/job_posting_%s.dat"


curFile = dataNameTmp%1
        
print(curFile," is Grouping Computing...")
datDf = read_dat_data(curFile)


    # count number of occurrences of each value in column '数据来源', generate a new column to record the count
datDf['count'] = datDf.groupby('数据来源')['数据来源'].transform('count')

    # drop duplicates based on column '数据来源', only keep the first occurrence
datDf = datDf.drop_duplicates(subset=['数据来源'], keep='first').reset_index(drop=True)
datDf = datDf[['数据来源', 'count']]
datDf
        

In [ ]:
df = pd.read_csv('F:/Data/job_posting/processed/estimation/charac_posting.csv', encoding = "utf_8_sig", on_bad_lines='skip', usecols = ['工作名称'])
df_new = df[df['工作名称'] == "轰动全国＋不压工资＋免费招聘＋仓管测试包装普工品管"]
df_new

In [ ]:
df = pd.read_csv('D:/job_posting/mapped_job_posting/job_res_110.csv', header=None, index_col=None, encoding='utf-8', sep="?", error_bad_lines=False, engine='python') # type: ignore

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
directory = 'F:/Data/job_posting/mapped_job_posting/Update file/'

# iterate over files in that directory
for filename in os.listdir(directory):
    # checking if it is a file
    if filename.startswith("job_res_"): # for files start with a prefix #
        f = os.path.join(directory, filename)
        df = pd.read_csv(f, encoding = "utf_8_sig", on_bad_lines='skip', delimiter= "?", header=None, encoding_errors='ignore')
        df.rename(columns={0: '招聘主键ID', 1: '公司ID', 2: '公司名称', 3: '城市名称', 4: '公司所在区域', 5: '工作薪酬', 6: '教育要求', 
                   7: '工作经历', 8: '工作描述', 9: '职位名称', 10: '工作名称', 11: '招聘数量', 12: '发布日期', 13: '行业名称', 
                   14: '来源'}, inplace=True)
        df_charac = df[['来源']]
        df_charac.to_csv('F:/Data/job_posting/processed/temp/{}'.format(filename))

In [2]:
import os
import openai
import pandas as pd
import re
import os
from glob import glob
from datetime import datetime
import gc
import requests
import json
import time
import numpy as np

In [3]:
# Append all the csv files under folder `E:/Data/job_posting/processed/title_mapped` to a single dataframe
path = r'F:/Data/job_posting/processed/title_mapped' # use your path
all_files = glob(os.path.join(path, "*.csv"))
# Append all the csv files in 'all_files' to a single dataframe
df_from_each_file = (pd.read_csv(f, encoding="utf_8_sig") for f in all_files)
df_title = pd.concat(df_from_each_file, ignore_index=True)
# subset dataframe 'df_title' to only keep '工作名称', 'soc_code'
df_title = df_title[['工作名称', 'soc_code']]
# drop the missing value in column 'soc_code'
df_title = df_title.dropna(subset=['soc_code'])
df_title

,工作名称,soc_code
0,销售代表,41-4012
1,销售经理,11-2022
2,会计,13-2011
3,销售专员,41-3099
4,电话销售,41-2022
...,...,...
883685,OA运维管理员,15-1142
883686,副总经理（分管行政）,11-1011
883689,调酒学员,35-3011
883691,正定政府保安,33-9032


In [15]:
directory = 'F:/Data/job_posting/mapped_job_posting/Update file/'

# create an empty DataFrame to store merged data
df_titleLabel = pd.DataFrame()

# iterate over files in that directory
for filename in os.listdir(directory):
    # checking if it is a file
    if filename.startswith("job_res_"): # for files start with a prefix #
        f = os.path.join(directory, filename)
        df = pd.read_csv(f, encoding = "utf_8_sig", on_bad_lines='skip', delimiter= "?", encoding_errors='ignore')
        df.rename(columns={'招聘ID': '招聘主键ID'}, inplace=True)
        df = df[['招聘主键ID', '工作描述', '工作名称']]
    # merge 'df_title' with 'df', based on column '工作名称'. Keep the matched row in csv file 'E:/Data/job_posting/processed/finetune/'
        df_titleData = pd.merge(df, df_title, on='工作名称', how='inner')# append the merged DataFrame to the empty DataFrame
        # change data type of 'soc_code' column to string
        # df_titleLabel['soc_code'] = df_titleLabel['soc_code'].astype(str)
        df_titleLabel = df_titleLabel.append(df_titleData)
        df_titleLabel['soc_code'] = df_titleLabel['soc_code'].astype(str)

# save the final merged DataFrame to a csv file
df_titleLabel.to_csv('F:/Data/job_posting/processed/finetune/df_titleLabel.csv', index=False, encoding = "utf_8_sig", header=True, quoting=csv.QUOTE_NONNUMERIC)


c:\Users\DELL\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3444: DtypeWarning: Columns (9,11) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)
c:\Users\DELL\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3444: DtypeWarning: Columns (11) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)
c:\Users\DELL\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3444: DtypeWarning: Columns (11,13) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)
c:\Users\DELL\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3444: DtypeWarning: Columns (9) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)
c:\Users\DELL\anaconda3\lib\site-packages\IPython\core\interacti

In [14]:
df = pd.read_csv('F:/Data/job_posting/mapped_job_posting/Update file/job_res_9.csv', encoding = "utf_8_sig", on_bad_lines='skip', delimiter= "?", encoding_errors='ignore', nrows = 10)
df

,招聘ID,公司ID,公司名称,城市名称,公司所在区域,工作薪酬,教育要求,工作经历,工作描述,职位名称,工作名称,招聘数量,发布日期,行业名称,数据来源,date
0,9104347,5927045,天津群悦信息技术有限公司,天津,天津河西区,4K-6K,大专,10年以上,NaN,会计,总账会计,NaN,2019-07-17,NaN,智联招聘,2019-07
1,9104348,195199539,江苏中南物业服务有限公司上海第二分公司,海南,文昌,4K-6K,大专,1-3年,NaN,物业,会计,NaN,2019-07-17,NaN,智联招聘,2019-07
2,9104349,9887319,中智天津人力资源服务有限公司,天津,天津河西区,8K-10K,本科,5-10年,NaN,会计,会计主管,NaN,2019-07-17,NaN,智联招聘,2019-07
3,9104350,19389492,湖南建科工程项目管理有限公司,海南,文昌,6K-12K,大专,5-10年,NaN,电工,机电工程师,NaN,2019-07-17,NaN,智联招聘,2019-07
4,9104351,3861350,海南鸿宝医药化工有限公司,海南,文昌,4K-6K,学历不限,不限,NaN,保险销售,云南白药业务员,NaN,2019-07-17,NaN,智联招聘,2019-07
5,9104352,150773649,天津市宇凡启航人力资源有限公司,天津,天津河西区,3.5K-4K,学历不限,不限,NaN,销售代表,网络推广,NaN,2019-07-17,NaN,智联招聘,2019-07
6,9104353,121202550,天津市桑田梓地农业科技有限公司,天津,天津蓟县,4K-6K,大专,1-3年,NaN,保险销售,销售主管,NaN,2019-07-17,NaN,智联招聘,2019-07
7,9104354,15376820,天津市红色传播平面设计有限公司,天津,天津河西区,4K-6K,学历不限,不限,NaN,平面设计,地产广告客户代表,NaN,2019-07-17,NaN,智联招聘,2019-07
8,9104355,93731989,天津童悦城教育信息咨询有限公司,天津,天津蓟县,2K-4K,中专,1-3年,NaN,保险销售,海帆亲子游泳俱乐部,NaN,2019-07-17,NaN,智联招聘,2019-07
9,9104356,111754187,广州市昌松老年旅游服务有限公司,海南,文昌,6K-8K,大专,1-3年,NaN,保险销售,超市店长,NaN,2019-07-17,NaN,智联招聘,2019-07
